# Interactive Spine Annotation with Napari

This notebook enables granular annotation of dendritic spines.

**Troubleshooting Visibility**:
1.  **3D Mode**: Ensure you click the **cube icon** in the bottom left of Napari.
2.  **Zoom to Layer**: Right-click 'Neuron Mesh' in the layer list and select 'Reset view'.
3.  **Coordinate Fix**: We move the neuron to (0,0,0) to prevent clipping.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import napari
import caveclient
from meshparty import trimesh_io, skeletonize
from scipy.spatial import cKDTree
import warnings

# Force Napari to use PySide6
os.environ["QT_API"] = "pyside6"
%gui qt

DATASTACK = 'minnie65_public'
OUTPUT_CSV = 'spine_labels_v1.csv'
CACHE_DIR = 'meshes'

client = caveclient.CAVEclient(DATASTACK)
mm = trimesh_io.MeshMeta(cv_path=client.info.segmentation_source(), disk_cache_path=CACHE_DIR)

print(f"Connected to {DATASTACK}.")

Connected to minnie65_public.


In [2]:
def get_sample_neuron(limit=1):
    df = client.materialize.query_table('nucleus_detection_v0', limit=limit + 10)
    ids = df['pt_root_id'].unique().tolist()
    return [x for x in ids if x > 100000000000000000]

def process_neuron(root_id):
    print(f"Downloading mesh for {root_id}...")
    mesh = mm.mesh(seg_id=root_id)
    offset = mesh.vertices.min(axis=0)
    print(f"Mesh has {len(mesh.vertices)} vertices.")
    
    sk = None
    print("Skeletonizing...")
    try:
        sk = skeletonize.skeletonize_mesh(mesh, invalidation_d=12000, compute_radius=False)
    except Exception as e:
        warnings.warn(f"Skeletonization failed: {e}.")
    
    return mesh, sk, offset

def save_annotations(root_id, mesh, points_data, offset):
    if points_data is None or len(points_data) == 0:
        print("No points marked! Skipping save.")
        return
    
    print(f"Mapping {len(points_data)} points to nearest mesh vertices...")
    # Transform points back to global coordinates
    global_points = points_data + offset
    
    # Map to global mesh vertices
    tree = cKDTree(mesh.vertices)
    distances, vertex_indices = tree.query(global_points, k=1)
    
    new_data = pd.DataFrame({
        'root_id': root_id,
        'vertex_index': vertex_indices,
        'label': 'spine',
        'distance_to_click': distances
    })
    
    if os.path.exists(OUTPUT_CSV):
        df = pd.read_csv(OUTPUT_CSV)
        df = pd.concat([df, new_data], ignore_index=True)
    else:
        df = new_data
    df.drop_duplicates(subset=['root_id', 'vertex_index'], keep='last', inplace=True)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Saved spine annotations to {OUTPUT_CSV}")

In [3]:
candidates = get_sample_neuron(limit=1)
# target_id = candidates[0]
target_id = 864691135724233643
mesh, skel, offset = process_neuron(target_id)

viewer = napari.Viewer(title=f"Annotating {target_id}")
viewer.dims.ndisplay = 3

local_vertices = (mesh.vertices - offset).astype(np.float32)
viewer.add_surface(
    (local_vertices, mesh.faces), 
    name='Neuron Mesh', 
    opacity=0.9, 
    shading='smooth',
    colormap='gray'
)

if skel is not None:
    local_skel = (skel.vertices - offset).astype(np.float32)
    viewer.add_points(
        local_skel,
        size=100, 
        face_color='blue', 
        name='Skeleton', 
        visible=False
    )

spines_layer = viewer.add_points(None, size=1500, face_color='red', name='Spines', ndim=3)
spines_layer.mode = 'add'

viewer.reset_view()

print("\n--- CONTROLS ---")
print("1. Rotate: Right-click and drag.")
print("2. Scale/Zoom: Scroll wheel.")
print("3. Reset View: Click the 'Reset View' icon (bottom left).")
print("4. ADD SPINES: Use the (+) button in the Spines layer controls.")
print("   - NOTE: If points appear 'floating' or 'back', don't worry!")
print("   - The Save script automatically SNAPS them to the nearest Mesh Vertex.")
print("Once done, CLOSE the window to continue to save.")

napari.run()

save_annotations(target_id, mesh, spines_layer.data, offset)

201 - "Limited query to 11 rows


Mesh has 279295 vertices.
Skeletonizing...


C:\Users\bkrou\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\meshparty\skeletonize.py:622: RuntimeWarning: invalid value encountered in multiply
  target = np.nanargmax(root_ds * valid)



--- CONTROLS ---
1. Rotate: Right-click and drag.
2. Scale/Zoom: Scroll wheel.
3. Reset View: Click the 'Reset View' icon (bottom left).
4. ADD SPINES: Use the (+) button in the Spines layer controls.
   - NOTE: If points appear 'floating' or 'back', don't worry!
   - The Save script automatically SNAPS them to the nearest Mesh Vertex.
Once done, CLOSE the window to continue to save.
No points marked! Skipping save.
